## Module 6-4o Explainable AI: Using `SHAP` to Understand an XGBoost Model

Parker et al. (2025) predict *future* material misstatements with an XGBoost classifier, then use an Explainable Artificial Intelligence (XAI) technique called **SHAP** (SHapley Additive exPlanations; Lundberg and Lee 2017) to identify which predictors matter most and how each one pushes a given prediction up or down.

6-1 ended by fitting an XGBoost model to predict the *direction of next-year earnings changes* and plotting its gain-based `feature_importances_`. That plot is a fast, model-level summary, but it can't tell us:

- Which direction a feature pushes a prediction (higher risk of an increase, or lower?).
- How that relationship might be nonlinear or depend on other features.
- Why the model made *this specific* prediction for *this specific* firm-year.

SHAP answers all three. We'll compute it for the exact same earnings-direction model from 5-1, and use the same suite of SHAP plots Parker et al. (2025) use for material misstatements — global importance, directional summary plots, dependence plots, and a single-observation waterfall explanation.

## Learning objectives

By the end of this class, you will be able to:

- Explain what a SHAP value is and how it differs from a gain-based feature-importance score.
- Use `shap.TreeExplainer` to compute SHAP values for an `xgboost.XGBClassifier`.
- Read a SHAP bar plot (global importance) and a SHAP beeswarm plot (directional importance).
- Use a SHAP dependence (scatter) plot to spot nonlinear or interaction effects.
- Use a SHAP waterfall plot to explain a single model prediction, decision by decision.
- Critically interpret SHAP output as an *attention-directing* tool rather than evidence of causation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
import shap

pd.set_option('display.max_columns', 50)

## 1. Recap

This is the same data pipeline, drift-adjusted label, feature engineering, chronological split, and early-stopped `XGBClassifier` as 6-2 — condensed into a few cells. Nothing here is new; skip ahead to Section 2 once it finishes running.

In [ ]:
df = pd.read_csv('data/comp_sample.csv')

df = df[
    (df['indfmt'] == 'INDL') &
    (df['curcd'] == 'USD') &
    (df['costat'] == 'A')
].copy()

df = df.dropna(subset=['gvkey', 'fyear']).copy()
df['fyear'] = df['fyear'].astype(int)
df = df.sort_values(['gvkey', 'fyear']).reset_index(drop=True)

# Drift-adjusted direction-of-earnings-change label (see 5-1, Section 2)
df['eps'] = (df['ni'] / df['csho']).replace([np.inf, -np.inf], np.nan)
df['d_eps'] = df.groupby('gvkey')['eps'].diff()
df['drift'] = (
    df.groupby('gvkey')['d_eps']
      .transform(lambda s: s.rolling(window=4, min_periods=1).mean())
)
df['d_eps_lead'] = df.groupby('gvkey')['d_eps'].shift(-1)
df['adj_d_eps_lead'] = df['d_eps_lead'] - df['drift']
df['label'] = np.where(
    df['adj_d_eps_lead'].isna(), np.nan, (df['adj_d_eps_lead'] > 0).astype(float)
)

print(df.shape)

In [ ]:
# Feature engineering: current value, lagged value, and % change for every
# auto-detected financial-statement column (see 5-1, Section 3)
ID_COLS = ['gvkey', 'datadate', 'fyear']
LABEL_COLS = ['eps', 'd_eps', 'drift', 'd_eps_lead', 'adj_d_eps_lead', 'label']
NON_FS_COLS = ['au']
NO_SCALE_COLS = ['at', 'csho', 'prcc_f']

exclude_cols = set(ID_COLS + LABEL_COLS + NON_FS_COLS)
numeric_cols = df.select_dtypes(include='number').columns
predictor_base_cols = [c for c in numeric_cols if c not in exclude_cols]


def build_features(data: pd.DataFrame, base_cols: list[str]) -> pd.DataFrame:
    at_cur = data['at']
    at_lag = data.groupby('gvkey')['at'].shift(1)

    feature_cols = {}
    for col in base_cols:
        cur = data[col]
        lag = data.groupby('gvkey')[col].shift(1)

        if col in NO_SCALE_COLS:
            cur_scaled, lag_scaled = cur, lag
        else:
            cur_scaled = (cur / at_cur).replace([np.inf, -np.inf], np.nan)
            lag_scaled = (lag / at_lag).replace([np.inf, -np.inf], np.nan)

        pct_change = ((cur - lag) / lag.abs()).replace([np.inf, -np.inf], np.nan)

        feature_cols[f'{col}_cur'] = cur_scaled
        feature_cols[f'{col}_lag'] = lag_scaled
        feature_cols[f'{col}_pctchg'] = pct_change

    return pd.DataFrame(feature_cols, index=data.index)


X_all = build_features(df, predictor_base_cols)
feature_cols = X_all.columns.tolist()

model_df = pd.concat([df[['gvkey', 'fyear', 'label']], X_all], axis=1)
model_df = model_df.dropna(subset=['label']).reset_index(drop=True)
print(model_df.shape)

In [ ]:
# Chronological train/validation/test split (see 5-1, Section 4)
TRAIN_END = 2019
VAL_END = 2021

train = model_df[model_df['fyear'] <= TRAIN_END]
val = model_df[(model_df['fyear'] > TRAIN_END) & (model_df['fyear'] <= VAL_END)]
test = model_df[model_df['fyear'] > VAL_END]

X_train, y_train = train[feature_cols], train['label']
X_val, y_val = val[feature_cols], val['label']
X_test, y_test = test[feature_cols], test['label']

print(f'train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}')

In [ ]:
# Early-stopped XGBoost classifier (see 5-1, Section 6.2) -- this is the
# model we will explain with SHAP for the rest of this notebook.
xgb_tuned = xgb.XGBClassifier(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    importance_type='gain',
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=42,
)
xgb_tuned.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f'Stopped after {xgb_tuned.best_iteration + 1} trees '
      f'(validation AUC = {xgb_tuned.best_score:.3f})')

## 2. What is a SHAP value?

SHAP is grounded in cooperative game theory (Shapley 1953): treat each feature as a "player" contributing to a "payout" (the model's prediction), and fairly split that payout across the players based on their marginal contributions across all possible orderings.

For any single prediction, SHAP decomposes the gap between the model's output for that observation, $f(x)$, and the model's average output across the sample, $E[f(x)]$, into one contribution per feature:

$$f(x) - E[f(x)] = \sum_{j} \phi_j$$

where $\phi_j$ is feature $j$'s SHAP value for that observation. A **positive** $\phi_j$ pushes the prediction above the average (here: toward predicting an earnings *increase*); a **negative** $\phi_j$ pushes it below average (toward a *decrease*). The larger $|\phi_j|$, the bigger that feature's role in *this* prediction.

This is exactly what a gain-based importance score cannot give us: gain is a single number per feature, aggregated across the whole model, with no sign and no per-observation detail (Parker et al. 2025, Section II).

We use `shap.TreeExplainer`, which computes *exact* SHAP values for tree ensembles like XGBoost in polynomial time (Lundberg et al. 2020) — much faster than the model-agnostic `KernelExplainer`, and exact rather than approximate.

In [ ]:
explainer = shap.TreeExplainer(xgb_tuned)
shap_explanation = explainer(X_test)

print(type(shap_explanation))
print('SHAP values shape (observations x features):', shap_explanation.values.shape)
print('Base value E[f(x)] (log-odds):', shap_explanation.base_values[0])

## 3. Global feature importance: SHAP vs. gain

Following Parker et al. (2025, Figure 2), we first rank predictors by the **mean absolute SHAP value** across the test set — a global importance measure, but built up from individual, signed contributions rather than a single training-time statistic. `shap.plots.bar` plots exactly this.

In [ ]:
shap.plots.bar(shap_explanation, max_display=20, show=False)
plt.tight_layout()
plt.show()

In [ ]:
# Compare directly to 5-1's gain-based ranking
gain_importance = pd.Series(
    xgb_tuned.feature_importances_, index=feature_cols
).sort_values(ascending=False)

shap_importance = pd.Series(
    np.abs(shap_explanation.values).mean(axis=0), index=feature_cols
).sort_values(ascending=False)

comparison = pd.DataFrame({
    'gain_rank': gain_importance.rank(ascending=False).astype(int),
    'shap_rank': shap_importance.rank(ascending=False).astype(int),
})
comparison.loc[shap_importance.head(10).index]

The two rankings usually agree on most of the top predictors but rarely match exactly — gain rewards features that produce large loss-function improvements when a split is *added*, even if that split affects few observations, whereas mean |SHAP| rewards features that meaningfully move *many* individual predictions. Large gaps between the two ranks are worth a closer look.

## 4. Directional importance: the SHAP beeswarm plot

A bar plot only shows *how much* each feature matters, on average. `shap.plots.beeswarm` also shows *which direction* and *for which observations*, mirroring Parker et al. (2025, Figure 3). Each dot is one test-set firm-year; its horizontal position is that feature's SHAP value for that observation, and its color is the (relatively) high (red) or low (blue) raw value of the feature itself.

In [ ]:
shap.plots.beeswarm(shap_explanation, max_display=15, show=False)
plt.tight_layout()
plt.show()

## 5. Which features push toward predicting an *increase*?

Following the same logic as Parker et al. (2025, Table 5), we can sort features by the **sum of their positive SHAP values** across the test set. A large positive sum means a feature frequently and substantially pushes predictions *up* — i.e., toward forecasting an earnings increase — whenever it takes the values it does in our data.

In [ ]:
positive_shap_sum = pd.Series(
    np.where(shap_explanation.values > 0, shap_explanation.values, 0).sum(axis=0),
    index=feature_cols,
).sort_values(ascending=False)

positive_shap_sum.head(10).rename('Sum of positive SHAP values').to_frame()

## 6. A closer look: SHAP dependence (scatter) plots

A dependence plot shows one feature's raw value on the x-axis and its SHAP value on the y-axis — the *directional impact* plots from Parker et al. (2025, Figure 3), but for a single feature at a time, which makes nonlinearities and interaction effects easier to read than in the beeswarm plot. `shap.plots.scatter` automatically colors points by whichever other feature the model relies on most for interaction, just like the two-way relationships Chen et al. (2022, Figure 7) visualize with partial dependence plots.

We plot it for the single most important feature identified in Section 3.

In [ ]:
top_feature = shap_importance.index[0]
print(f'Most important feature by mean |SHAP|: {top_feature}')

shap.plots.scatter(shap_explanation[:, top_feature], show=False)
plt.tight_layout()
plt.show()

## 7. Explaining a single prediction: the SHAP waterfall plot

So far every plot has summarized the *whole* test set. A waterfall plot does the opposite: it explains **one** prediction, feature by feature, exactly like the Aerojet Rocketdyne Holdings example in Parker et al. (2025, Appendix D).

We pick the test-set firm-year our model is *most confident* will see an earnings increase, and show how each feature's SHAP value adds up, starting from the average log-odds $E[f(x)]$, to the model's final log-odds $f(x)$ for that firm-year.

In [ ]:
pred_proba = xgb_tuned.predict_proba(X_test)[:, 1]
most_confident_idx = np.argmax(pred_proba)

firm_info = test.iloc[most_confident_idx]
print(f"gvkey {int(firm_info['gvkey'])}, fiscal year {int(firm_info['fyear'])}")
print(f'Predicted probability of an earnings increase: {pred_proba[most_confident_idx]:.3f}')

shap.plots.waterfall(shap_explanation[most_confident_idx], max_display=12, show=False)
plt.tight_layout()
plt.show()

The bottom of the plot, $E[f(x)]$, is the average predicted log-odds across the test set. Each bar moves us toward this specific firm-year's predicted log-odds, $f(x)$, at the top. Red bars push the prediction toward *increase*; blue bars push it toward *decrease*. Summing every feature's contribution (plus the small residual from features not shown) recovers $f(x)$ exactly — the same accounting identity underlying Parker et al.'s Appendix D example.

## 8. Wrap-up and discussion

**A caution, straight from the paper:** Parker et al. (2025, Section V) are explicit that predictive features "are not necessarily the root causes... they should be used as attention-directing risk indicators that warrant further investigation" (p. 239). The same applies here — a feature with a large SHAP value tells us the *model* leaned on it heavily for a prediction, not that it *causes* future earnings changes. SHAP explains the model, not the world.

**What SHAP added, on top of Module 6-2's gain plot:**

- **Direction**: whether a feature pushes toward *increase* or *decrease*, not just how much it matters overall (Section 4).
- **Nonlinearity**: dependence plots can reveal thresholds or interaction effects a single gain score can't (Section 6).
- **Instance-level explanations**: a defensible, decomposable answer to "why did the model predict *this* for *this* firm-year?" (Section 7) — the kind of explanation an analyst, auditor, or regulator could actually act on.